In [4]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DecimalType
from pyspark.sql.functions import create_map, lit   

In [ ]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_at = dbutils.widgets.get("processed_at")
catalog = dbutils.widgets.get("catalog")

In [7]:
schema = StructType([
            StructField("ride_id", StringType(), True),
            StructField("rideable_type", StringType(), True),
            StructField("started_at", TimestampType(), True),
            StructField("ended_at", TimestampType(), True),
            StructField("start_station_name", StringType(), True),
            StructField("start_station_id", StringType(), True),
            StructField("end_station_name", StringType(), True),
            StructField("end_station_id", StringType(), True),
            StructField("start_lat", DecimalType(), True),
            StructField("start_lng", DecimalType(), True),
            StructField("end_lat", DecimalType(), True),
            StructField("end_lng", DecimalType(), True),
            StructField("member_casual", StringType(), True)

        ]
    )

In [ ]:
df = spark.read.csv((f"/Volumes/{catalog}/00_landing/source_file"), header=True, schema=schema)

In [ ]:
df = df.withColumn(
    "metadata", 
    create_map(
        lit("pipeline_id"), lit(pipeline_id),
        lit("run_id"), lit(run_id),
        lit("task_id"), lit(task_id),
        lit("processed_at"), lit(processed_at)
    )
)



,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,metadata
0,29DAF43DD84B4B7A,electric_bike,2025-03-20 18:58:31.217,2025-03-20 19:00:46.466,6 St & Grand St,HB302,Mama Johnson Field - 4 St & Jackson St,HB404,41,-74,41,-74,member,"{'source_file': '/Volumes/citibike_dev/00_landing/source_file', 'run_id': 'some_run_id', 'task_id': 'some_task_id', 'processed_at': 'some_timestamp'}"
1,B11B4220F7195025,electric_bike,2025-03-29 11:01:25.124,2025-03-29 11:11:09.383,Heights Elevator,JC059,Jersey & 3rd,JC074,41,-74,41,-74,member,"{'source_file': '/Volumes/citibike_dev/00_landing/source_file', 'run_id': 'some_run_id', 'task_id': 'some_task_id', 'processed_at': 'some_timestamp'}"
2,18D5B30305F602B9,electric_bike,2025-03-01 16:05:32.346,2025-03-01 16:07:43.156,Jersey & 3rd,JC074,Hamilton Park,JC009,41,-74,41,-74,member,"{'source_file': '/Volumes/citibike_dev/00_landing/source_file', 'run_id': 'some_run_id', 'task_id': 'some_task_id', 'processed_at': 'some_timestamp'}"
3,532EB2D9DB68567D,electric_bike,2025-03-21 18:44:15.137,2025-03-21 18:51:00.763,Jersey & 3rd,JC074,Jersey & 6th St,JC027,41,-74,41,-74,member,"{'source_file': '/Volumes/citibike_dev/00_landing/source_file', 'run_id': 'some_run_id', 'task_id': 'some_task_id', 'processed_at': 'some_timestamp'}"
4,EA7C9C945D7D57AA,electric_bike,2025-03-20 11:08:27.226,2025-03-20 11:12:28.545,6 St & Grand St,HB302,Madison St & 1 St,HB402,41,-74,41,-74,member,"{'source_file': '/Volumes/citibike_dev/00_landing/source_file', 'run_id': 'some_run_id', 'task_id': 'some_task_id', 'processed_at': 'some_timestamp'}"


In [ ]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.01_bronze.citibike")
)